# Convolutional Neural Network (Handwriting Recognition)
Convolutional neural networks imitate the structure of neurons that process images in the brain and use techniques to reduce neuron count while maintaining positional relationships in the data.

This is a convolutional network for the MNIST data set to recognize handwritten digits. In addition, you'll analyze how the convolutional neural network works differently than the densely connected network. You'll be able to feed images of handwritten digits to the model and it should recognize it. For example, you can draw down a digit on paper and then scan it, then upload as an image. If this seems too cumbersome, you're right. There's actually a quickdraw webcam model also in this repository that's much easier to use, based on the same model. (The folder name, is: "webcam_drawpad_model".)

# Data Preparation
To prepare your data, you'll import the libraries, load the MNIST dataset, format the data, and then perform one-hot encoding. For this network, you'll follow the process from the previous lessons.

## Importing TensorFlow and Keras
To get started, you'll import the TensorFlow and Keras libraries. In addition, you'll also import the NumPy and Matplotlib libraries, including the inline command.

In [1]:
# Import TensorFlow and Keras to create the neural network.
import tensorflow as tf
from tensorflow import keras

# Import the MNIST dataset and Keras backend.
from tensorflow.keras.datasets import mnist
from tensorflow.keras import backend as K

# Import the NumPy and Matplotlib libraries and add the inline command.
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


## Load the Data
Now that you've set up the functions and the variables, you'll load the data to train the model and test its accuracy. In addition, you'll also print the shape of the training and test image datasets.

In [ ]:
# Load the MNIST Data
def show_min_max(array, i):
    random_image = array[i]
    print("minimum and maximum pixel values in image: ", random_image.min(), random_image.max())

In [ ]:
# Create a function which will plot a image from the dataset and display the image.
def plot_image(array, i, labels):
    plt.imshow(np.squeeze(array[i])) # show the image and convert shape
    plt.title(" Digit " + str(labels[i]))
    plt.xticks([])
    plt.yticks([])
    plt.show()

In [ ]:
# Create variables for the image row and column to keep track of your image size.
img_rows, img_cols = 28, 28

# Create a variable called num_classes and set the value to 10 output classes.
num_classes = 10

In [ ]:
# Load the data to train and test the model, as well as the labels to test the data against.
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Load a backup copy of the untouched data, while the first copy will be processing the data and manipulating it.
(train_images_backup, train_labels_backup), (test_images_backup, test_labels_backup) = mnist.load_data()

# Print the shape of the training image dataset.
print(train_images.shape)

# Print the shape of the test image dataset.
print(test_images.shape)

(60000, 28, 28)
(10000, 28, 28)


## Data Formatting & One-Hot Encoding
For now, you'll reshape the data to be an appropriate size for your network. In addition, you'll employ one-hot encoding to replace the label on each image with a vectorized representation for the network to process.

In [ ]:
# Reshape the training data by converting the list of pixels into a 28x28 grid.
train_images = train_images.reshape(train_images.shape[0], img_rows, img_cols, 1)


# Reshape the test data by converting the list of pixels into a 28x28 grid.
test_images = test_images.reshape(test_images.shape[0], img_rows, img_cols, 1)

# Create an input_shape variable to keep track of the data's shape.
input_shape = (img_rows, img_cols, 1)

# Change the image values to between 0 and 1 by converting the training and test data into float32.
train_images = train_images.astype('float32')
test_images = test_images.astype('float32')

# Divide the images by 255 to make sure that each pixel is stored as a value between 0 and 1.
train_images /= 255
test_images /= 255

# Employ one-hot encoding on your training labels.
train_labels = keras.utils.to_categorical(train_labels, num_classes)

# Employ one-hot encoding on your test labels.
test_labels = keras.utils.to_categorical(test_labels, num_classes)

# Print the shape of the training data.
print(train_images[1232].shape)

(28, 28, 1)


# Building the Network
Like with the densely connected network, you're going to set up epochs. Remember, epochs are the rounds for training the network or how many times it should pass over the data.

You'll also define this network as **Sequential()**. Convolutional networks, like densely connected ones, take the output from one layer to feed in as input in the next layer. However, this network uses different types of layers.

## Import Model and Layers
To get started, you'll import the **Sequential** model and the **Dense**, **Flatten**, **Conv2D**, **MaxPooling2D**, and **Dropout** layers.

In the previous network that you created, you used the Dense, Flatten, and Dropout layers. In this network, you'll also use the **Conv2D**, **MaxPooling2D**, and **Dropout** layers.

Here is a breakdown of what each layer does:


*   The **Conv2D()** layer processes two-dimensional data, such as images, and creates a convolutional layer.
*   The **MaxPooling2D()** layer reduces the size of each data piece to keep the important information.
*   The **Dropout()** layer ensures the network isn't trained on a particular pattern. Therefore, the Dropout() layer ignores a certain amount of data at different iterations. For example, Dropout(0.3) randomly ignores 30% of neurons during an iteration.



In [ ]:
# Import the Sequential model.
from tensorflow.keras.models import Sequential

# Import the Dense, Flatten, Conv2D, MaxPooling2D, and Dropout layers.
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout

## Epochs and Model Type
Next, you'll set the epochs and model type. Every image in the dataset is passed through your model once. Generally, the more epochs you run, the better your results, but the longer it will take to train. Finding the balance between good results and reasonable runtime is a big challenge in training a model. To start, you'll run 10 epochs.

In [ ]:
# Create a variable called epochs and set the value as 10.
epochs = 10

# Create a new model object using the Keras Sequential command.
model = Sequential()

# Adding Layers
You added three fully connected (or dense) layers to your densely connected network. The layers in this network are convolutional and work differently.

Convolutional layers use small clusters of neurons called filters that are moved across the image and activated based on the pixels they see. These clusters learn to recognize features in the data.

You can adjust the number and size of filters in the layer — larger filters observe larger areas of the image at once, while smaller filters spot finer details. A higher filter count will allow a wider range of features to be recognized.

There are multiple advantages to having filters that work this way:


*   Small filters are more computationally efficient since they only examine a small portion of the image at once.
*   Just like in real life, the filter can do a better job by focusing on a small simple problem and ignoring the distraction of the rest of the image.
*   Because the filters are moved across the entire image, convolutional networks are good at identifying two images that have the same object but in different places in each image.





## Implementing Convolutional Layers
Keras provides functionality to easily create convolutional layers for your neural networks. You'll use the function Conv2D to create the first convolutional layer of your network.

In [ ]:
# Add a Conv2D layer to the network.
model.add(Conv2D(filters=32, kernel_size=(3,3), activation='relu', input_shape=input_shape))

## Pooling Layers
Processing images with convolutional layers can get computationally complex. Successive layers of convolutions increase the number of neurons and computation time required.

**Pooling layers** are used by convolutional networks to manage the growth of complexity by simplifying and shrinking the dataset.

Pooling layers use a filter that moves across the data with a specified **stride**, simplifying the contents of each filter into a single value. This shrinks the size of the layer's output based on the filter's size.

This also helps reduce the network's **translation variance**, which is how sensitive the network is to an object's exact position in an image.

The most common pooling layer is a 2x2 filter with a stride of 2. This reduces the width and height of the input layer by half, simplifying the data without too much loss of specificity in the image.

## Adding a Pooling Layer
Keras uses the MaxPooling2D function to create 2D pooling layers.

In [ ]:
# Add a MaxPooling2D layer to the network.
model.add(MaxPooling2D(pool_size=(2,2)))

## More Convolutional Layers
By design, convolutional layers can examine an image's low-level features. By adding more convolutional layers, the network can start to work with higher-level features.

This layer is defined the same way as the last one, except for more filters, 64, whereas the previous one had 32. Also, the input shape doesn't need to be defined since it's inferred from previous layer.

In [ ]:
# Add another Conv2D layer to the network.
model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu'))

## Dropout Layers
A **dropout layer** takes a percentage of all the neurons in the input and deactivates them at random. This random dropout of neurons forces more of the network to adapt to the task.

Without a dropout layer, larger networks run the risk of growing overdependent on a small set of competent neurons rather than the whole network learning the task.

In [ ]:
# Add a Dropout layer to the network.
model.add(Dropout(rate=0.3))

# Add the final calculation layer, Conv2D layer to the network.
model.add(Conv2D(32, (3,3), activation='relu'))

## Dense and Flatten Layers
At the end of the convolutional and pooling layers, you'll set up some neurons to help make your final classification decision. This will be a standard, fully connected layer of neurons. You'll first flatten the 2D image's filters to connect these layers.

Keras uses the **Flatten** function to create a flattening layer and the **Dense** function to create dense layers. In this case, the **Dense** layer will have an output size of 32 neurons.



In [ ]:
# Add a Flatten() layer to the network.
model.add(Flatten())

# Add a Dense layer to the network.
model.add(Dense(units=32, activation='relu'))

## Output Layers
Just like with your fully connected network, the final layer needs to shrink the previous layer down to just the number of possible classes. Like before, the decision is represented by the class with the highest weight.

In [ ]:
# Add another Dense layer to the network.
model.add(Dense(units=10, activation='softmax'))

# Print a summary of your network.
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_9 (Conv2D)           (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 13, 13, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_10 (Conv2D)          (None, 11, 11, 64)        18496     
                                                                 
 dropout_1 (Dropout)         (None, 11, 11, 64)        0         
                                                                 
 conv2d_11 (Conv2D)          (None, 9, 9, 32)          18464     
                                                                 
 flatten_1 (Flatten)         (None, 2592)              0         
                                                      

# Training the Network
Just like with your first network, you'll compile this network. The loss and metrics will be the same as your last network: categorical_crossentropy and accuracy. This time, however, you'll use a different training algorithm called **RMSProp**.

**RMSProp** is one of several different training algorithms Keras defines to do the computation that teaches the network how to improve.

The neural network aims to optimize the loss by making it as small as possible. RMSProp is one way the network can do this.

## Compile the Network
To compile the network, you'll set the optimizer as RMSProp. In addition, you'll use the categorical cross-entropy algorithm to calculate the loss and get the accuracy.



In [ ]:
# Add the compile function that calculates the loss and uses the optimizer parameter to set the optimization algorithm.
model.compile(loss='categorical_crossentropy', optimizer='rmsprop', metrics=['accuracy'])


## Training
The **fit()** function does the actual work of running the training. Now that the data has been processed and the model has been defined and compiled, it's time to train the model based on the data. In this network, you're going to use validation data to get an idea of how the network is performing with each epoch instead of just once at the end.

In [ ]:
# Add the fit function and set the input data for the model so the network doesn't rely on a pattern to learn.
model.fit(train_images, train_labels, batch_size=64, epochs=epochs, validation_data=(test_images, test_labels), shuffle=True)

Epoch 1/10
938/938 [==============================] - 25s 26ms/step - loss: 0.1666 - accuracy: 0.9482 - val_loss: 0.0465 - val_accuracy: 0.9851
Epoch 2/10
938/938 [==============================] - 28s 30ms/step - loss: 0.0512 - accuracy: 0.9843 - val_loss: 0.0325 - val_accuracy: 0.9891
Epoch 3/10
938/938 [==============================] - 27s 29ms/step - loss: 0.0364 - accuracy: 0.9887 - val_loss: 0.0303 - val_accuracy: 0.9899
Epoch 4/10
938/938 [==============================] - 26s 28ms/step - loss: 0.0280 - accuracy: 0.9915 - val_loss: 0.0260 - val_accuracy: 0.9916
Epoch 5/10
938/938 [==============================] - 26s 28ms/step - loss: 0.0224 - accuracy: 0.9928 - val_loss: 0.0273 - val_accuracy: 0.9913
Epoch 6/10
938/938 [==============================] - 26s 28ms/step - loss: 0.0201 - accuracy: 0.9939 - val_loss: 0.0309 - val_accuracy: 0.9905
Epoch 7/10
938/938 [==============================] - 26s 28ms/step - loss: 0.0156 - accuracy: 0.9953 - val_loss: 0.0263 - val_accuracy:

## Evaluation and Returning the Model
Now that you have created your model, compiled it, and trained it, you'll test it and see how accurate it is on data it has yet to see! Just like with your other network, you'll need to evaluate your network.

The **evaluate()** function returns an object that stores the evaluation results. You'll add a few arguments to this function so the network knows what data to test itself on.

Like the other stages, Tensorflow has some tools to help you out.

In [2]:
# Calculate the loss and accuracy of your model.
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
scores = model.evaluate(test_images, test_labels, verbose=0)

# Print out the test accuracy.
print('Test accuracy:', scores[1])

NameError: name 'model' is not defined

## Exporting the Model
Like before, you'll save a copy of this trained model to use later.

In [ ]:
# Export your model.
model.save('cnn_model.h5')

C:\Users\Student\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
# Predict the Image
def predict_image(model, i):
    i = i.astype('float32')
    i = np.expand_dims(i,axis=0)
    image_predict = model.predict(i, verbose=0)
    print("Predicted Label: ", np.argmax(image_predict))
    plt.imshow(np.squeeze(i))
    plt.xticks([])
    plt.yticks([])
    plt.show()
    return image_predict

# Dinosaur
def plot_value_array(predictions_array, true_label, h):
    plt.grid(False)
    plt.xticks(range(10))
    plt.yticks([])
    this_plot = plt.bar(range(10), predictions_array[0], color="#777777")
    plt.ylim([(-1*h),h])
    predicted_label = np.argmax(predictions_array)
    this_plot[predicted_label].set_color('red')
    this_plot[true_label].set_color('blue')
    plt.plot()

model = tf.keras.models.load_model('cnn_model.h5')

path = "9-test-inverted.jpg"
img = tf.keras.preprocessing.image.load_img(path, target_size=(28,28), color_mode="grayscale")
x = tf.keras.preprocessing.image.img_to_array(img)

# Run the densely connect network to see the prediction for the image.
true_label = 9
p_arr = predict_image(model, x)
plot_value_array(p_arr, true_label, 1)

NameError: name 'tf' is not defined